In [18]:
# Sel 1: Import libraries
import re
import matplotlib.pyplot as plt
import logging
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Konfigurasi logging
logging.basicConfig(level=logging.INFO)

In [19]:
# Sel 2: Load data
logging.info("1. Memuat data...")
columns = ['Harga_Normalized', 'Kamar_Normalized', 'WC_Normalized', 'Parkir_Normalized',
           'Luas_Tanah_Normalized', 'Luas_Bangunan_Normalized', 'Judul_Clean', 'Lokasi_Clean',
           'Deskripsi_Clean', 'Image_Link', 'Property_Link']
df = pd.read_csv('databaru.csv', encoding='utf-8')
df = df[columns]
print(df.head())
print(df.info())



INFO:root:1. Memuat data...


   Harga_Normalized  Kamar_Normalized  WC_Normalized  Parkir_Normalized  \
0          0.664804          0.111111            0.2                  0   
1          0.469274          0.222222            0.2                  1   
2          0.449721          0.222222            0.2                  0   
3          0.208101          0.222222            0.1                  0   
4          0.092179          0.111111            0.0                  0   

   Luas_Tanah_Normalized  Luas_Bangunan_Normalized  \
0               0.456522                  0.464115   
1               0.778261                  0.607656   
2               0.378261                  0.464115   
3               0.369565                  0.368421   
4               0.256522                  0.157895   

                                         Judul_Clean         Lokasi_Clean  \
0  rumah baru mewah patra siap huni angsur 79x flatt       ngaglik sleman   
1     jual rumah dekat kampus stie ykpn harga rendah  caturtunggal sle

In [20]:
# Sel 3: Preprocessing data
logging.info("2. Mempersiapkan fitur dan target...")

# Gabungkan teks yang sudah dibersihkan
df['text_combined'] = df['Judul_Clean'] + ' ' + \
                      df['Lokasi_Clean'] + ' ' + \
                      df['Deskripsi_Clean']

# Hitung panjang dokumen
df['doc_length'] = df['text_combined'].str.len()

# Filter dokumen dengan panjang > 3 karakter
df = df[df['text_combined'].str.strip().str.len() > 3]

print("\nJumlah dokumen setelah preprocessing:", len(df))


INFO:root:2. Mempersiapkan fitur dan target...



Jumlah dokumen setelah preprocessing: 7656


In [21]:
# Sel 4: TF-IDF Vectorization (diperbaiki: fit dulu baru simpan, tidak duplikat di sel lain)
logging.info("3. Melakukan TF-IDF Vectorization (single step)...")

# Inisialisasi dan fit TF-IDF
TFIDF_MAX_FEATURES = 1000
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, min_df=1, stop_words=None)
try:
    text_features = tfidf.fit_transform(df['text_combined']).toarray()
except ValueError as e:
    print("\n[ERROR] Terjadi error dalam TF-IDF Vectorization:", e)
    print("\n[DEBUG] Contoh isi text_combined (5 dokumen pertama):")
    print(df['text_combined'].head())
    print("\n[DEBUG] Distribusi panjang dokumen:")
    print(df['text_combined'].str.len().describe())
    raise

# Simpan vectorizer SETELAH fit
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("\n[DEBUG] Fitur TF-IDF berhasil dibuat:")
print("Shape:", text_features.shape)
print("Contoh fitur (baris pertama, 10 dimensi pertama):", text_features[0][:10])


INFO:root:3. Melakukan TF-IDF Vectorization (single step)...



[DEBUG] Fitur TF-IDF berhasil dibuat:
Shape: (7656, 1000)
Contoh fitur (baris pertama, 10 dimensi pertama): [0.         0.         0.         0.         0.         0.
 0.0652176  0.10292151 0.         0.        ]


In [22]:
# Sel 5: Prepare features and target
logging.info("4. Mempersiapkan fitur dan target...")
features = text_features
target = df['Harga_Normalized'].values

# Hapus sample yang memiliki NaN
mask = ~np.isnan(target)
features = features[mask]
target = target[mask]



INFO:root:4. Mempersiapkan fitur dan target...


In [23]:
# Sel 6: Split data
logging.info("5. Membagi data latih dan uji...")
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

INFO:root:5. Membagi data latih dan uji...


In [24]:
# Sel 7: Persiapan Fitur Numerik (refactor: hilangkan duplikasi TF-IDF, cegah leakage, tangani division by zero)
logging.info("6. Menyusun fitur numerik...")

# Fitur dasar numerik (TANPA Harga_Normalized untuk hindari leakage)
base_numeric = df[[
    'Kamar_Normalized', 'WC_Normalized', 'Parkir_Normalized',
    'Luas_Tanah_Normalized', 'Luas_Bangunan_Normalized'
]].copy()

# Tangani nilai 0 untuk rasio: ganti 0 menjadi np.nan sementara supaya tidak ada inf
luas_bangunan_safe = base_numeric['Luas_Bangunan_Normalized'].replace(0, np.nan)
luas_tanah_safe = base_numeric['Luas_Tanah_Normalized'].replace(0, np.nan)

price = df['Harga_Normalized']  # target asli
# Rekayasa fitur rasio (akan menghilangkan baris yang menghasilkan NaN nanti)
price_per_bangunan = price / luas_bangunan_safe
bangunan_per_tanah = luas_bangunan_safe / luas_tanah_safe
kamar_wc_mul = base_numeric['Kamar_Normalized'] * base_numeric['WC_Normalized']

numeric_features = np.column_stack([
    base_numeric.values,
    price_per_bangunan,            # hubungan harga per luas bangunan (boleh sebagai fitur turunan)
    bangunan_per_tanah,            # rasio efisiensi lahan
    kamar_wc_mul                   # interaksi kamar & WC
])

numeric_feature_names = [
    'Kamar_N', 'WC_N', 'Parkir_N', 'Luas_Tanah_N', 'Luas_Bangunan_N',
    'Harga_per_LuasBangunan', 'LuasBangunan_per_LuasTanah', 'Kamar_x_WC'
]

# Target
target = price.values

# Validasi & bersihkan baris yang mengandung NaN/inf (drop serempak di semua set)
valid_mask = np.isfinite(numeric_features).all(axis=1) & np.isfinite(target)
removed = (~valid_mask).sum()
if removed > 0:
    logging.warning(f"Menghapus {removed} baris yang mengandung NaN/inf pada fitur numerik atau target.")

numeric_features = numeric_features[valid_mask]
text_features = text_features[valid_mask]
target = target[valid_mask]

print("\n[DEBUG] Fitur numerik selesai:")
print("Shape:", numeric_features.shape)
print("Contoh fitur (baris pertama):", numeric_features[0])
print("Jumlah target:", target.shape)

# Pemeriksaan tambahan sebelum split
print("Cek NaN text_features:", np.isnan(text_features).any())
print("Cek NaN numeric_features:", np.isnan(numeric_features).any())
print("Cek Inf numeric_features:", np.isinf(numeric_features).any())
print("Rentang target:", float(np.nanmin(target)), float(np.nanmax(target)))


INFO:root:6. Menyusun fitur numerik...



[DEBUG] Fitur numerik selesai:
Shape: (7654, 8)
Contoh fitur (baris pertama): [0.11111111 0.2        0.         0.45652174 0.46411483 1.43241375
 1.01663249 0.02222222]
Jumlah target: (7654,)
Cek NaN text_features: False
Cek NaN numeric_features: False
Cek Inf numeric_features: False
Rentang target: 0.0 1.0


In [25]:
# Sel 8: Split data
logging.info("7. Membagi data latih/val/uji...")
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    text_features, numeric_features, target, test_size=0.2, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_text_train, X_num_train, y_train, test_size=0.2, random_state=42
)
print("Ukuran set:")
print("Text -> train/val/test:", X_text_train.shape, X_text_val.shape, X_text_test.shape)
print("Num  -> train/val/test:", X_num_train.shape, X_num_val.shape, X_num_test.shape)


INFO:root:7. Membagi data latih/val/uji...


Ukuran set:
Text -> train/val/test: (4898, 1000) (1225, 1000) (1531, 1000)
Num  -> train/val/test: (4898, 8) (1225, 8) (1531, 8)


In [26]:
# Sel 9: Create and train text model
logging.info("8. Membuat dan melatih model teks...")
from tensorflow.keras.callbacks import TerminateOnNaN
text_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_text_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])

text_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

text_history = text_model.fit(
    X_text_train, y_train, 
    validation_data=(X_text_val, y_val),
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stopping, TerminateOnNaN()],
    verbose=1
)
print("Contoh loss text (5 epoch pertama):", text_history.history['loss'][:5])


INFO:root:8. Membuat dan melatih model teks...


Epoch 1/100
154/154 [==============================] - 1s 4ms/step - loss: 0.0295 - mae: 0.1212 - val_loss: 0.0147 - val_mae: 0.0885
Epoch 2/100
154/154 [==============================] - 1s 4ms/step - loss: 0.0295 - mae: 0.1212 - val_loss: 0.0147 - val_mae: 0.0885
Epoch 2/100
154/154 [==============================] - 0s 3ms/step - loss: 0.0125 - mae: 0.0787 - val_loss: 0.0113 - val_mae: 0.0718
Epoch 3/100
154/154 [==============================] - 0s 3ms/step - loss: 0.0125 - mae: 0.0787 - val_loss: 0.0113 - val_mae: 0.0718
Epoch 3/100
154/154 [==============================] - 0s 3ms/step - loss: 0.0094 - mae: 0.0676 - val_loss: 0.0109 - val_mae: 0.0694
Epoch 4/100
154/154 [==============================] - 0s 3ms/step - loss: 0.0094 - mae: 0.0676 - val_loss: 0.0109 - val_mae: 0.0694
Epoch 4/100
154/154 [==============================] - 0s 2ms/step - loss: 0.0072 - mae: 0.0596 - val_loss: 0.0109 - val_mae: 0.0653
Epoch 5/100
154/154 [==============================] - 0s 2ms/step - 

In [27]:
# Sel 10: Create and train numeric model
logging.info("9. Membuat dan melatih model numerik...")
from tensorflow.keras.callbacks import TerminateOnNaN
numeric_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_num_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)
])

numeric_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

numeric_history = numeric_model.fit(
    X_num_train, y_train, 
    validation_data=(X_num_val, y_val),
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stopping, TerminateOnNaN()],
    verbose=1,
)
print("Contoh loss numeric (5 epoch pertama):", numeric_history.history['loss'][:5])


INFO:root:9. Membuat dan melatih model numerik...


Epoch 1/100
154/154 [==============================] - 1s 3ms/step - loss: 0.0177 - mae: 0.0949 - val_loss: 0.0189 - val_mae: 0.0984
Epoch 2/100
154/154 [==============================] - 1s 3ms/step - loss: 0.0177 - mae: 0.0949 - val_loss: 0.0189 - val_mae: 0.0984
Epoch 2/100
154/154 [==============================] - 0s 2ms/step - loss: 0.0068 - mae: 0.0578 - val_loss: 0.0096 - val_mae: 0.0723
Epoch 3/100
154/154 [==============================] - 0s 2ms/step - loss: 0.0068 - mae: 0.0578 - val_loss: 0.0096 - val_mae: 0.0723
Epoch 3/100
154/154 [==============================] - 0s 2ms/step - loss: 0.0059 - mae: 0.0480 - val_loss: 0.0058 - val_mae: 0.0574
Epoch 4/100
154/154 [==============================] - 0s 2ms/step - loss: 0.0059 - mae: 0.0480 - val_loss: 0.0058 - val_mae: 0.0574
Epoch 4/100
154/154 [==============================] - 0s 2ms/step - loss: 0.0047 - mae: 0.0446 - val_loss: 0.0077 - val_mae: 0.0710
Epoch 5/100
154/154 [==============================] - 0s 2ms/step - 

In [28]:
# Sel 11: Evaluate models & Plot (robust)
logging.info("10. Mengevaluasi & mem-plot model...")

def safe_plot(ax, history, train_key, val_key, title, y_label):
    train_vals = history.history.get(train_key, [])
    val_vals = history.history.get(val_key, [])
    ax.set_title(title)
    ax.set_xlabel('Epochs')
    ax.set_ylabel(y_label)
    if len(train_vals) > 0:
        ax.plot(train_vals, label=f'Train {train_key}')
    if len(val_vals) > 0:
        ax.plot(val_vals, label=f'Val {val_key}')
    if len(train_vals)==0 and len(val_vals)==0:
        ax.text(0.5,0.5,'(no data)', ha='center', va='center', transform=ax.transAxes, color='red')
    ax.legend()

plt.figure(figsize=(14,5))
ax1 = plt.subplot(1,2,1)
safe_plot(ax1, text_history, 'loss', 'val_loss', 'Text Model Loss', 'MSE')
ax2 = plt.subplot(1,2,2)
safe_plot(ax2, text_history, 'mae', 'val_mae', 'Text Model MAE', 'MAE')
plt.tight_layout()
plt.savefig('text_model_training_performance.png')
plt.close()

plt.figure(figsize=(14,5))
ax1 = plt.subplot(1,2,1)
safe_plot(ax1, numeric_history, 'loss', 'val_loss', 'Numeric Model Loss', 'MSE')
ax2 = plt.subplot(1,2,2)
safe_plot(ax2, numeric_history, 'mae', 'val_mae', 'Numeric Model MAE', 'MAE')
plt.tight_layout()
plt.savefig('numeric_model_training_performance.png')
plt.close()

# Predictions & scatter
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_text = text_model.predict(X_text_test).flatten()
y_pred_numeric = numeric_model.predict(X_num_test).flatten()

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title('Text Model: Actual vs Predicted')
plt.scatter(y_test, y_pred_text, alpha=0.5)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn,mx],[mn,mx],'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')

plt.subplot(1,2,2)
plt.title('Numeric Model: Actual vs Predicted')
plt.scatter(y_test, y_pred_numeric, alpha=0.5)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn,mx],[mn,mx],'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.tight_layout()
plt.savefig('model_prediction_comparison.png')
plt.close()

# Metrics
metrics = {}
for name, y_pred in [('text', y_pred_text), ('numeric', y_pred_numeric)]:
    metrics[name] = {
        'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred))),
        'MAE': float(mean_absolute_error(y_test, y_pred)),
        'R2': float(r2_score(y_test, y_pred))
    }
print("Metrics:", metrics)
print("Visualization images saved:")
print("- text_model_training_performance.png")
print("- numeric_model_training_performance.png")
print("- model_prediction_comparison.png")


INFO:root:10. Mengevaluasi & mem-plot model...


48/48 [==============================] - 0s 1ms/step
Metrics: {'text': {'RMSE': 0.08382710865730328, 'MAE': 0.05482833207936219, 'R2': 0.820642360090175}, 'numeric': {'RMSE': 0.03729948177538315, 'MAE': 0.030823102707530258, 'R2': 0.9644895232702191}}
Visualization images saved:
- text_model_training_performance.png
- numeric_model_training_performance.png
- model_prediction_comparison.png
Metrics: {'text': {'RMSE': 0.08382710865730328, 'MAE': 0.05482833207936219, 'R2': 0.820642360090175}, 'numeric': {'RMSE': 0.03729948177538315, 'MAE': 0.030823102707530258, 'R2': 0.9644895232702191}}
Visualization images saved:
- text_model_training_performance.png
- numeric_model_training_performance.png
- model_prediction_comparison.png


In [29]:
# Sel 12: Save models & metadata
logging.info("11. Menyimpan model & metadata...")
text_model.save('text_model.keras')
numeric_model.save('numeric_model.keras')

# Simpan nama fitur numerik untuk referensi deployment
with open('numeric_feature_names.txt','w', encoding='utf-8') as f:
    for n in numeric_feature_names:
        f.write(n+'\n')

logging.info("Proses pelatihan model selesai.")


INFO:root:11. Menyimpan model & metadata...
INFO:root:Proses pelatihan model selesai.
INFO:root:Proses pelatihan model selesai.


In [30]:
# Sel 13: Save models h5
logging.info("9. Menyimpan model...")
text_model.save('text_model.h5')
numeric_model.save('numeric_model.h5')

print("Models have been saved as text_model.h5 and numeric_model.h5")

INFO:root:9. Menyimpan model...


Models have been saved as text_model.h5 and numeric_model.h5


c:\Users\galan\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
